In [ ]:
import pandas as pd
import requests
from io import StringIO

# DataSF API
url = "https://data.sfgov.org/resource/nuek-vuh3.csv"

# for our Project we made an interval as fisrt we wanted to study the year statsics & the website Link will change 1/9/2026.
# to make the analysis dynamic escpcially for python notebooks' Analysis all you need to remove the upper limit to get the up-to-date data.
# then load the fresh Data in Data Preprocessing notebook and continue the workflow.
params = {
    "$where": """
        received_dttm >= '2025-09-01T00:00:00'
        AND received_dttm < '2026-08-27T00:00:00'
    """,
    "$limit": 500000
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

response.raise_for_status()

df = pd.read_csv(StringIO(response.text), low_memory=False)

print("Rows:", len(df))
print("Columns:", len(df.columns))

df["received_dttm"] = pd.to_datetime(
    df["received_dttm"],
    errors="coerce"
)

df["date"] = df["received_dttm"].dt.date

print("Start:", df["received_dttm"].min())
print("End:", df["received_dttm"].max())
print("Unique dates:", df["date"].nunique())

Status code: 200
Rows: 359542
Columns: 36
Start: 2025-09-01 00:00:46
End: 2026-08-26 23:56:48
Unique dates: 360


In [ ]:
import requests
import pandas as pd

# San Francisco coordinates
latitude = 37.77
longitude = -122.42

# Date range from our dataset
start_date = df["received_dttm"].min().strftime("%Y-%m-%d")
end_date = df["received_dttm"].max().strftime("%Y-%m-%d")

print("Weather period:", start_date, "to", end_date)

# Open-Meteo Archive API
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": start_date,
    "end_date": end_date,
    "daily": [
        "temperature_2m_mean",
        "temperature_2m_min",
        "temperature_2m_max",
        "precipitation_sum",
        "wind_speed_10m_max"
    ],
    "timezone": "America/San_Francisco"
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

response.raise_for_status()

weather_data = response.json()

print("Weather data received successfully!")

Weather period: 2025-09-01 to 2026-08-26
Status code: 200
Weather data received successfully!


In [ ]:
weather_df = pd.DataFrame({
    "date": pd.to_datetime(weather_data["daily"]["time"]).date,
    "avg_temperature": weather_data["daily"]["temperature_2m_mean"],
    "min_temperature": weather_data["daily"]["temperature_2m_min"],
    "max_temperature": weather_data["daily"]["temperature_2m_max"],
    "precipitation": weather_data["daily"]["precipitation_sum"],
    "max_wind_speed": weather_data["daily"]["wind_speed_10m_max"]
})

print("Weather rows:", len(weather_df))
print("\nWeather columns:")
print(weather_df.columns.tolist())

print("\nFirst 5 days:")
display(weather_df.head())

Weather rows: 360

Weather columns:
['date', 'avg_temperature', 'min_temperature', 'max_temperature', 'precipitation', 'max_wind_speed']

First 5 days:


,date,avg_temperature,min_temperature,max_temperature,precipitation,max_wind_speed
0,2025-09-01,19.8,15.4,25.4,0.0,15.3
1,2025-09-02,18.0,16.1,21.0,0.0,19.9
2,2025-09-03,16.6,14.5,20.0,0.0,17.6
3,2025-09-04,16.1,14.2,18.7,0.1,21.9
4,2025-09-05,16.5,14.8,19.1,0.0,23.1


In [ ]:
# Install holidays library if needed
!pip install holidays -q

import holidays
import pandas as pd

# Get the years in our dataset
years = range(
    df["received_dttm"].dt.year.min(),
    df["received_dttm"].dt.year.max() + 1
)

# US Federal + California holidays
us_ca_holidays = holidays.US(
    years=years,
    subdiv="CA"
)

# Create holidays DataFrame
holiday_df = pd.DataFrame({
    "date": list(us_ca_holidays.keys()),
    "holiday_name": list(us_ca_holidays.values())
})

holiday_df["date"] = pd.to_datetime(
    holiday_df["date"]
).dt.date

print("Number of holidays:", len(holiday_df))
print("\nSample holidays:")
display(holiday_df.head(15))

Number of holidays: 28

Sample holidays:


,date,holiday_name
0,2025-01-01,New Year's Day
1,2025-05-26,Memorial Day
2,2025-06-19,Juneteenth National Independence Day
3,2025-07-04,Independence Day
4,2025-09-01,Labor Day
5,2025-11-27,Thanksgiving Day
6,2025-12-25,Christmas Day
7,2025-01-20,Martin Luther King Jr. Day
8,2025-11-11,Veterans Day
9,2025-02-17,Presidents' Day


In [ ]:
# Merge weather data with the emergency calls dataset
df = df.merge(
    weather_df,
    on="date",
    how="left"
)

# Merge holiday data
df = df.merge(
    holiday_df,
    on="date",
    how="left"
)

# Create holiday flag
df["is_holiday"] = df["holiday_name"].notna()

print("Final rows:", len(df))
print("Final columns:", len(df.columns))

print("\nMissing weather values:")
print(df[
    [
        "avg_temperature",
        "min_temperature",
        "max_temperature",
        "precipitation",
        "max_wind_speed"
    ]
].isna().sum())

print("\nHoliday counts:")
print(df["is_holiday"].value_counts())

Final rows: 359542
Final columns: 44

Missing weather values:
avg_temperature    0
min_temperature    0
max_temperature    0
precipitation      0
max_wind_speed     0
dtype: int64

Holiday counts:
is_holiday
False    346103
True      13439
Name: count, dtype: int64


In [ ]:
output_file = "sf_ems_weather_merged.csv"

df.to_csv(output_file, index=False)

print("File saved successfully!")
print("File name:", output_file)
print("Rows:", len(df))
print("Columns:", len(df.columns))

File saved successfully!
File name: sf_ems_weather_merged.csv
Rows: 359542
Columns: 44


In [ ]:
from google.colab import drive
import shutil
import os

# Connect Google Drive
drive.mount('/content/drive')

# Copy the final file to Google Drive
source = "/content/sf_ems_weather_merged.csv"
destination = "/content/drive/MyDrive/sf_ems_weather_merged.csv"

shutil.copy2(source, destination)

print("File copied successfully!")
print("Saved to:", destination)
print("File size:", round(os.path.getsize(destination) / (1024**2), 2), "MB")

Mounted at /content/drive
File copied successfully!
Saved to: /content/drive/MyDrive/sf_ems_weather_merged.csv
File size: 174.84 MB
